# 01 Runs: pilot, then the resumable matrix

Every finished run appends one line to `results/runs.jsonl` on Drive. Re-running a cell after a disconnect skips what is done.

In [ ]:
# Setup: mount Drive and import the project code.
# Put the project folder (src/, configs/) at MyDrive/paper1/code, or set REPO_URL to a git remote.
from google.colab import drive
drive.mount("/content/drive")

import logging, subprocess, sys
REPO_URL = ""  # optional
CODE = "/content/drive/MyDrive/paper1/code"
if REPO_URL:
    CODE = "/content/zdssl"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=False)
    subprocess.run(["git", "-C", CODE, "pull"], check=True)
sys.path.insert(0, f"{CODE}/src")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

from zdssl.config import config_hash, load_config, resolve_paths
cfg = load_config(f"{CODE}/configs/paper1.yaml")
paths = resolve_paths(cfg)
print("code:", CODE, "| results:", paths["runs"])


In [ ]:
import torch
from zdssl.runner import build_matrix, run_stage
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, slow)")


In [ ]:
# Pilot: every model once (seed 42, 10% budget). Measures wall time and extrapolates.
import pandas as pd
from zdssl.report import flatten, load_runs
result = run_stage(cfg, "pilot")
print("failed:", result["failed"])
pilot_keys = {s.key for s in build_matrix(cfg, "pilot")}
pilot = [r for r in load_runs(paths["runs"], config_hash(cfg)) if r["key"] in pilot_keys]
timing = pd.DataFrame([{"model": r["model"], "wall_s": r["timing"]["wall_seconds"],
                        "pretrain_s": r["timing"].get("pretrain_seconds", 0.0)} for r in pilot])
per_seed = timing["wall_s"].sum()
print(timing.to_string(index=False))
print(f"\nestimate for the 240-run matrix: about {per_seed * 40 / 60:.0f} min "
      "(upper bound: pretraining is paid once per seed, not once per budget)")


In [ ]:
# Priority 1: 10% budget, 10 seeds, all models (Tables 3, 4, 5A).
result = run_stage(cfg, "p1"); print({k: (len(v) if isinstance(v, list) else v) for k, v in result.items()})


In [ ]:
# Priority 2: the other budgets (Fig. 2).
result = run_stage(cfg, "p2"); print({k: (len(v) if isinstance(v, list) else v) for k, v in result.items()})


In [ ]:
# Priority 3 and 4: objective ablation (Table 5B) and Condition A (shortcut check).
for stage in ("ablation", "condA"):
    result = run_stage(cfg, stage); print({k: (len(v) if isinstance(v, list) else v) for k, v in result.items()})


In [ ]:
# Status: what is done, and any failures (full tracebacks are in results/errors.jsonl).
from zdssl.runner import completed_keys
done = completed_keys(paths["runs"], config_hash(cfg))
for stage in ("p1", "p2", "ablation", "condA"):
    specs = build_matrix(cfg, stage)
    print(f"{stage:9s} {sum(s.key in done for s in specs):3d} / {len(specs)}")
if paths["errors"].is_file():
    print("\nerrors logged:"); print(paths["errors"].read_text()[-3000:])
